## Structured Data
parsing excel and csv files

In [4]:
import pandas as pd
import os

In [3]:
os.makedirs("data/structured_files", exist_ok=True)

In [7]:
# Create sample data
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    'Stock': [50, 200, 150, 75, 100],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}
# Save as CSV
df = pd.DataFrame(data)
df.to_csv('data/structured_files/products.csv', index=False)

## Save as excelfile with multiple sheets

In [9]:
with pd.ExcelWriter('data/structured_files/inventory.xlsx') as writer:
  df.to_excel(writer, sheet_name='Products', index=False)
  # Add another sheet with summary stats
  summary_data = {
    'Category': ['Electronics', 'Accessories'],
    'Total_Items': [3, 2],
    'Total_Value': [1389.97, 109.98]
  }
  pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

# CSV Processing

In [10]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

/home/valac/Projects/RAG/RAG-Udemi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Method 1: CSVLoader -  Each Row becomes a Document

In [11]:
print("1: CSVLoaders - Row-based Documents")
csv_loader = CSVLoader(
  file_path='data/structured_files/products.csv', 
  encoding='utf-8', 
  csv_args={'delimiter': ',', 'quotechar': '"'}
  )

csv_docs = csv_loader.load()
print(f"Loaded {len(csv_docs)} documents (one per row)")
print("\nFirst document")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")

1: CSVLoaders - Row-based Documents
Loaded 5 documents (one per row)

First document
Content: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/structured_files/products.csv', 'row': 0}


In [12]:
csv_docs

[Document(metadata={'source': 'data/structured_files/products.csv', 'row': 0}, page_content='Product: Laptop\nCategory: Electronics\nPrice: 999.99\nStock: 50\nDescription: High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row': 1}, page_content='Product: Mouse\nCategory: Accessories\nPrice: 29.99\nStock: 200\nDescription: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row': 2}, page_content='Product: Keyboard\nCategory: Accessories\nPrice: 79.99\nStock: 150\nDescription: Mechanical keyboard with RGB backlighting'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row': 3}, page_content='Product: Monitor\nCategory: Electronics\nPrice: 299.99\nStock: 75\nDescription: 27-inch 4K monitor with HDR support'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row': 4}, page_content='Product: Webcam\nCategory: Ele

In [2]:
from typing import List
from langchain_core.documents import Document

# Method 2: Custom CSV processing for better control
print("\n2: Custom CSV Processing")
def process_csv_intelligently(file_path: str) -> List[Document]:
  """Process CSV with intelligent document creation"""
  df = pd.read_csv(file_path)
  documents = []

  # Strategy 1: Create one document per row with structured content
  for idx, row in df.iterrows():
    # Create structured content
    content = f"""Procduct Information:
    Name: {row['Product']}
    Category: {row['Category']}
    Price: ${row['Price']}
    Stock: {row['Stock']}
  """
    doc = Document(
      page_content = content,
      metadata = {
        'source': file_path,
        'row_index': idx,
        'product': row['Product'],
        'category': row['Category'],
        'price': row['Price'],
        'data_type': 'product_info'
      }
    )
    documents.append(doc)
  return documents


2: Custom CSV Processing


In [5]:
process_csv_intelligently('data/structured_files/products.csv')

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'data_type': 'product_info'}, page_content='Procduct Information:\n    Name: Laptop\n    Category: Electronics\n    Price: $999.99\n    Stock: 50\n  '),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'data_type': 'product_info'}, page_content='Procduct Information:\n    Name: Mouse\n    Category: Accessories\n    Price: $29.99\n    Stock: 200\n  '),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product': 'Keyboard', 'category': 'Accessories', 'price': 79.99, 'data_type': 'product_info'}, page_content='Procduct Information:\n    Name: Keyboard\n    Category: Accessories\n    Price: $79.99\n    Stock: 150\n  '),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 3, 'product':

# Excel Processing

In [7]:


def process_excel_with_pandas(file_path: str) -> List[Document]:
  """Process Excel with sheet awareness and structured content"""
  documents = []

  # Read all sheet
  excel_file = pd.ExcelFile(file_path)

  for sheet_name in excel_file.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)

    # Create document for each sheet
    sheet_content = f"Sheet: {sheet_name}\n"
    sheet_content += f"Columns: {', '.join(df.columns)}\n"
    sheet_content += f"Rows: {len(df)}\n\n"
    sheet_content += df.to_string(index=False)

    doc = Document(
      page_content=sheet_content,
      metadata={
        'source': file_path,
        'sheet_name': sheet_name,
        'num_rows': len(df),
        'num_columns': len(df.columns),
        'data_type': 'excel_sheet'
      }
    )

    documents.append(doc)
  return documents



In [8]:
excel_docs = process_excel_with_pandas('data/structured_files/inventory.xlsx')
print(f"Loaded {len(excel_docs)} documents (one per sheet)")
print("\nFirst document")
print(f"Content: {excel_docs[0].page_content[:500]}...")
print(f"Metadata: {excel_docs[0].metadata}")

Loaded 2 documents (one per sheet)

First document
Content: Sheet: Products
Columns: Product, Category, Price, Stock, Description
Rows: 5

 Product    Category  Price  Stock                                         Description
  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD
   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design
Keyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting
 Monitor Electronics 299.99     75                 27-inch 4K monitor wit...
Metadata: {'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}


In [13]:
from langchain_community.document_loaders import UnstructuredExcelLoader
# Method 2: Using UnstructuredExcelLoader for more complex Excel files
print("\n3: UnstructuredExcelLoader for Excel Files")
try:
  excel_loader = UnstructuredExcelLoader(
    file_path='data/structured_files/inventory.xlsx',
    mode="elements"
  )

  unstructured_docs = excel_loader.load()
  print("Handles complex excel features")
  print("Preserves formatting and structure info ")
  print("Requires unstructured library and dependencies")
except Exception as e:
  print(f"Error loading Excel with UnstructuredExcelLoader: {e}")


3: UnstructuredExcelLoader for Excel Files
Handles complex excel features
Preserves formatting and structure info 
Requires unstructured library and dependencies


In [14]:
unstructured_docs

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-03-01T11:47:24', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>Keyboard</td><td>Accessories</td><td>79.99</td><td>150</td><td>Mechanical keyboard with RGB backlighting</td></tr><tr><td>Monitor</td><td>Electronics</td><td>299.99</td><td>75</td><td>27-inch 4K monitor with HDR support</td></tr><tr><td>Webcam</td><td>Electronics</td><td>89.99</td><td>100</td><td>1080p webcam with noise cancellation</td></tr></table>', 'languages': ['eng'], 'filetype': 'applicatio